In [ ]:
import torch
print(torch.__version__)       # e.g.: 2.1.0
print(torch.version.cuda)      # e.g.: 11.8

In [ ]:
!pip install  huggingface_hub

!pip install imagecorruptions
!pip install --upgrade scikit-image

In [ ]:
!pip install mmcv==2.1.0 -q

# Verify
import mmcv
print(mmcv.__version__)

In [ ]:
import os

# Environment path for imagecorruptions package
file_path = '/usr/local/lib/python3.12/dist-packages/imagecorruptions/corruptions.py'

if os.path.exists(file_path):
    # Read file contents
    with open(file_path, 'r') as file:
        file_data = file.read()

    # Patch deprecated parameter calls
    file_data = file_data.replace('multichannel=True', 'channel_axis=-1')
    file_data = file_data.replace('multichannel=False', 'channel_axis=None')
    
    # Overwrite patched file
    with open(file_path, 'w') as file:
        file.write(file_data)
        
    print("✅ Successfully patched corruptions.py via Python!")
else:
    print("❌ File not found. Please verify the file path.")

> **Note**: Execute environment setup cells above once upon fresh kernel initialization, then restart runtime if dependencies were updated.


In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
from pprint import pprint
import pickle
import numpy as np
np.float_ = np.float64

from pycocotools import mask as mask_util
from PIL import Image
import albumentations as A
import random
import torch
import os
import cv2

In [ ]:
def set_seed(seed=42):
    """
    Fix all random sources to ensure reproducible benchmark results.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Multi-GPU support
    
    # Ensure deterministic cuDNN algorithms
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f">>> Deterministic Seed set: {seed}")

# Initialize seed
set_seed(42)

In [ ]:
import os
from huggingface_hub import login

# Authenticate with Hugging Face Hub securely
# Priority: 1. Kaggle Secrets -> 2. Environment Variable -> 3. Interactive prompt
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("✅ Successfully logged in to Hugging Face Hub via secret/env token.")
else:
    login()  # Prompts securely in notebook without storing plain-text token


In [ ]:
import numpy as np
import random
from PIL import Image
from imagecorruptions import corrupt, get_corruption_names

class NoiseFactory:
    """
    Standard Benchmark Perturbation Factory (ImageNet-C):
    - Based on imagecorruptions library (15 standard types + 4 extra)
    - Input: PIL Image or Numpy Array
    - Output: PIL Image
    """

    # Supported corruptions from imagecorruptions library
    SUPPORTED_NOISES = get_corruption_names()

    @staticmethod
    def set_seed(seed=42):
        """Ensure deterministic execution across the entire benchmark."""
        random.seed(seed)
        np.random.seed(seed)

    @classmethod
    def apply(cls, image, noise_type, severity=1):
        """
        Core entry point. 
        Accepts PIL or Numpy -> Converts to Numpy -> Calls corruptions library -> Returns PIL.
        """
        # 1. Verify corruption type validity
        if noise_type not in cls.SUPPORTED_NOISES:
            raise ValueError(f"Unsupported corruption type '{noise_type}'. "
                             f"Valid types include: {', '.join(cls.SUPPORTED_NOISES)}")

        # 2. Convert input (PIL / numpy -> uint8 RGB numpy array)
        if isinstance(image, Image.Image):
            # Ensure RGB color format before numpy conversion
            img_np = np.array(image.convert("RGB"))
        elif isinstance(image, np.ndarray):
            img_np = image.copy()
            # imagecorruptions expects uint8 input
            if img_np.dtype != np.uint8:
                img_np = img_np.astype(np.uint8)
        else:
            raise TypeError(f"Input must be a PIL Image or Numpy Array, received {type(image)}")
        
        # 3. Clamp severity level to range [1, 5]
        severity = max(1, min(severity, 5))

        # 4. Apply ImageNet-C benchmark corruption
        # imagecorruptions executes underlying perturbation
        noisy_np = corrupt(img_np, corruption_name=noise_type, severity=severity)

        # 5. Convert output (numpy -> PIL) and return
        return Image.fromarray(noisy_np)

    @classmethod
    def get_available_noises(cls):
        """Utility function to list all supported corruption types."""
        return cls.SUPPORTED_NOISES

In [ ]:
import cv2
import numpy as np
from PIL import Image

class AddNoise:
    def __init__(self, 
                 image_noise_type=None, image_severity=1, 
                 bbox_noise_type=None, bbox_severity=1, 
                 mask_noise_type=None, mask_severity=1):
        """
        Comprehensive 3-tier perturbation engine (Image - BBox - Mask) for benchmark evaluation.
        ...
        """
        self.image_noise_type = image_noise_type
        self.image_severity = image_severity

        self.bbox_noise_type = bbox_noise_type
        self.bbox_severity = bbox_severity

        self.mask_noise_type = mask_noise_type
        self.mask_severity = mask_severity

    # ==========================================
    # 1. IMAGE PERTURBATIONS (via NoiseFactory)
    # ==========================================
    def _apply_image_noise(self, image):
        if not self.image_noise_type:
            return image
            
        is_numpy = isinstance(image, np.ndarray)
        image_input = Image.fromarray(image) if is_numpy else image
            
        # Assumes NoiseFactory is defined
        noisy_pil = NoiseFactory.apply(
            image=image_input, 
            noise_type=self.image_noise_type, 
            severity=self.image_severity
        )

        return np.array(noisy_pil) if is_numpy else noisy_pil

    # ==========================================
    # 2. BOUNDING BOX SPATIAL PERTURBATIONS
    # ==========================================
    def xywh_to_xyxy(self, box):
        x, y, w, h = box
        return [x, y, x + w, y + h]

    # Extract bounding box [x, y, w, h] from binary mask
    def _extract_bbox_from_mask(self, mask):
        mask_np = np.array(mask)
        # Ensure 2D mask array
        if mask_np.ndim > 2:
            mask_np = mask_np[:, :, 0]
            
        y_indices, x_indices = np.where(mask_np > 0)
        
        # Handle empty mask edge case
        if len(y_indices) == 0 or len(x_indices) == 0:
            return [0, 0, 0, 0]
            
        xmin, xmax = np.min(x_indices), np.max(x_indices)
        ymin, ymax = np.min(y_indices), np.max(y_indices)
        
        w = xmax - xmin
        h = ymax - ymin
        
        return [int(xmin), int(ymin), int(w), int(h)]

    def _apply_bbox_noise(self, bboxes, W, H):
        if not getattr(self, 'bbox_noise_type', None):
            clean_bboxes = []
            for bbox in bboxes:
                x, y, w, h = bbox
                clean_bboxes.append(self.xywh_to_xyxy([int(x), int(y), int(w), int(h)]))
            return clean_bboxes          

        severity = max(1, min(getattr(self, 'bbox_severity', 1), 5))
        noisy_bboxes = []

        for bbox in bboxes:
            x, y, w, h = bbox
            cx, cy = x + w / 2, y + h / 2

            if self.bbox_noise_type == "shift":
                shift_ratio = severity * 0.05 
                dx = np.random.choice([-1, 1]) * (w * shift_ratio)
                dy = np.random.choice([-1, 1]) * (h * shift_ratio)
                cx, cy = cx + dx, cy + dy

            elif self.bbox_noise_type == "upscale":
                scale_ratio = 1.0 + (severity * 0.05)
                w, h = w * scale_ratio, h * scale_ratio

            elif self.bbox_noise_type == "downscale":
                scale_ratio = 1.0 - (severity * 0.05)
                w, h = w * scale_ratio, h * scale_ratio
                
            else:
                raise ValueError(f"Unsupported BBox perturbation: {self.bbox_noise_type}")

            nx = np.clip(cx - w / 2, 0, W - 1)
            ny = np.clip(cy - h / 2, 0, H - 1)
            nw = np.clip(w, 1, W - nx)
            nh = np.clip(h, 1, H - ny)         

            noisy_bboxes.append(self.xywh_to_xyxy([int(nx), int(ny), int(nw), int(nh)]))
            
        return noisy_bboxes

    
    # ==========================================
    # 3. MASK MORPHOLOGICAL PERTURBATIONS
    # ==========================================
    def _apply_mask_noise(self, mask, W, H):
        if not getattr(self, 'mask_noise_type', None):
            return mask
            
        mask_np = np.array(mask)
        is_binary_01 = mask_np.max() <= 1
        if is_binary_01:
            mask_np = (mask_np * 255).astype(np.uint8)
        else:
            mask_np = mask_np.astype(np.uint8)

        severity = max(1, min(getattr(self, 'mask_severity', 1), 5))
        
        if self.mask_noise_type in ["dilation", "erosion"]:
            # [SYNC FIX] Replaced harsh morphological kernels with Sub-pixel Rescaling
            # This perfectly matches the 5% BBox scaling logic and works on tiny objects.
            pos = np.where(mask_np > 0)
            
            # Safety: If mask is already empty, return it
            if pos[0].size == 0:
                noisy_mask = mask_np
            else:
                scale = 1.0 + (severity * 0.05) if self.mask_noise_type == "dilation" else 1.0 - (severity * 0.05)
                
                ymin, ymax = np.min(pos[0]), np.max(pos[0])
                xmin, xmax = np.min(pos[1]), np.max(pos[1])
                obj_w = xmax - xmin + 1
                obj_h = ymax - ymin + 1
                
                obj_crop = mask_np[ymin:ymax+1, xmin:xmax+1]
                
                # Calculate new size, ensuring it's at least 1x1 pixel
                new_w = max(1, int(round(obj_w * scale)))
                new_h = max(1, int(round(obj_h * scale)))
                
                rescaled_obj = cv2.resize(obj_crop, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
                rescaled_obj = (rescaled_obj > 127).astype(np.uint8) * 255
                
                noisy_mask = np.zeros_like(mask_np)
                
                # Keep it centered based on the original bounding box
                start_y = ymin + (obj_h - new_h) // 2
                start_x = xmin + (obj_w - new_w) // 2
                
                # Safety Case: Ensure we don't paste outside image boundaries
                end_y = min(start_y + new_h, H)
                end_x = min(start_x + new_w, W)
                start_y = max(0, start_y)
                start_x = max(0, start_x)
                
                slice_h = end_y - start_y
                slice_w = end_x - start_x
                
                if slice_h > 0 and slice_w > 0:
                    noisy_mask[start_y:end_y, start_x:end_x] = rescaled_obj[:slice_h, :slice_w]
                else:
                    # Fallback if resizing pushed the object entirely off-screen
                    noisy_mask = mask_np

        elif self.mask_noise_type == "translation":
            pos = np.where(mask_np > 0)
            
            # Safety: If mask is already empty, return it
            if pos[0].size == 0:
                noisy_mask = mask_np
            else:
                # 1. Get the exact bounds of the mask
                ymin, ymax = np.min(pos[0]), np.max(pos[0])
                xmin, xmax = np.min(pos[1]), np.max(pos[1])
                obj_w = xmax - xmin + 1
                obj_h = ymax - ymin + 1
                
                # 2. Match the bbox severity ratio (0.05)
                shift_ratio = severity * 0.05
                tx = obj_w * shift_ratio
                ty = obj_h * shift_ratio
                
                # 3. Roll the dice for direction
                dx = np.random.choice([-1, 1]) * tx
                dy = np.random.choice([-1, 1]) * ty
                
                # 4. [EXACT SAFETY MATCH] 
                # Calculate the maximum allowed shift before the mask hits the image boundary.
                # We use np.clip on dx and dy to pin the mask to the edge, 
                # exactly mirroring how the BBox uses np.clip on its coordinates.
                max_dx_left = -xmin
                max_dx_right = (W - 1) - xmax
                max_dy_up = -ymin
                max_dy_down = (H - 1) - ymax
                
                dx = np.clip(dx, max_dx_left, max_dx_right)
                dy = np.clip(dy, max_dy_up, max_dy_down)
                
                # 5. Apply the safe shift
                M = np.float32([[1, 0, dx], [0, 1, dy]])
                noisy_mask = cv2.warpAffine(mask_np, M, (W, H))
            
        elif self.mask_noise_type == "boundary_jitter":
            contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            noisy_mask = np.zeros_like(mask_np)
            
            jitter_amp = severity * 2 
            
            jittered_contours = []
            for cnt in contours:
                noise = np.random.randint(-jitter_amp, jitter_amp + 1, cnt.shape)
                jittered_cnt = cnt + noise
                
                # [SAFETY FIX] Clip X and Y independently against W and H
                jittered_cnt[..., 0] = np.clip(jittered_cnt[..., 0], 0, W - 1)  # Clip X
                jittered_cnt[..., 1] = np.clip(jittered_cnt[..., 1], 0, H - 1)  # Clip Y
                
                jittered_contours.append(jittered_cnt)
                
            cv2.drawContours(noisy_mask, jittered_contours, -1, 255, thickness=cv2.FILLED)

        else:
            raise ValueError(f"Unsupported mask perturbation: {self.mask_noise_type}")

        if is_binary_01:
            noisy_mask = (noisy_mask > 127).astype(np.uint8)

        return Image.fromarray(noisy_mask) if isinstance(mask, Image.Image) else noisy_mask


    # ==========================================
    # UNIFIED DISPATCH ENTRY POINT
    # ==========================================
    def apply_to_data(self, image, bboxes=None, masks=None):
        if isinstance(image, np.ndarray):
            H, W = image.shape[:2]
        else:
            W, H = image.size 
            
        # 1. Apply image noise
        noisy_img = self._apply_image_noise(image)
        
        # Temporary holder for current bounding boxes
        current_bboxes = bboxes

        # 2. Apply mask noise and derive updated bounding boxes
        noisy_masks = None
        if masks is not None:
            if isinstance(masks, list):
                noisy_masks = [self._apply_mask_noise(m, W, H) for m in masks]
                # Re-derive bounding box from perturbed mask
                current_bboxes = [self._extract_bbox_from_mask(m) for m in noisy_masks]
            else:
                noisy_masks = self._apply_mask_noise(masks, W, H)
                # Re-derive bounding box from perturbed mask
                current_bboxes = [self._extract_bbox_from_mask(noisy_masks)]
        
        # 3. Apply spatial prompt noise to bounding boxes
        noisy_bboxes = self._apply_bbox_noise(current_bboxes, W, H) if current_bboxes else None

        return noisy_img, noisy_bboxes, noisy_masks

In [ ]:
import ast
import json
import os
import pickle
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from pycocotools import mask as mask_util

class UniversalDataset(Dataset):
    def __init__(self, hf_dataset, dataset_name, perturber=None, local_path=None):
        self.hf_dataset = hf_dataset
        self.dataset_name = dataset_name.lower()
        self.perturber = perturber # AddNoise perturber instance
        self.local_path = local_path

        # Dispatcher for dataset format parsers
        self.parsers = {
            'lvis': self._parse_standard,
            'cocostuff': self._parse_standard,
            'mapillary': self._parse_standard,
            'openimages': self._parse_standard, 
            'paco': self._parse_paco,
            'dlc_bench': self._parse_dlc_bench,
            'flickr30k': self._parse_flickr30k
        }

        if self.dataset_name not in self.parsers:
            raise ValueError(f"Dataset '{self.dataset_name}' is not supported.")

        # DLC-Bench is read from local storage, not Hugging Face
        if self.dataset_name == 'dlc_bench':
            self._init_dlc_bench()

    def _init_dlc_bench(self):
        """
        Read annotations.json from local disk,
        build img_dict, cat_dict, and a complete sample list with all fields.
        """
        if not self.local_path:
            raise ValueError("DLC-Bench requires local_path pointing to dataset directory.")

        anno_path    = os.path.join(self.local_path, "annotations.json")
        image_folder = os.path.join(self.local_path, "images")

        if not os.path.exists(anno_path):
            raise FileNotFoundError(f"File not found: {anno_path}")

        print(f"-> Loading annotations from: {anno_path}")
        with open(anno_path, 'r', encoding='utf-8') as f:
            coco_data = json.load(f)

        # Mapping: image ID -> image file name
        self._img_dict = {img['id']: img['file_name'] for img in coco_data.get('images', [])}

        # Mapping: category ID -> category name
        self._cat_dict = {cat['id']: cat['name'] for cat in coco_data.get('categories', [])}

        # Iterate through annotations and retain original fields
        self.dlc_samples = []
        for ann in coco_data.get('annotations', []):
            img_id = ann['image_id']
            if img_id not in self._img_dict:
                continue

            file_name     = self._img_dict[img_id]
            img_path      = os.path.join(image_folder, file_name)
            category_name = self._cat_dict.get(ann.get('category_id'), 'Unknown')

            self.dlc_samples.append({
                # Identity
                "ann_id":       ann['id'],
                "img_id":       img_id,
                "image_path":   img_path,
                # Semantics
                "category":     category_name,
                "caption":      category_name,
                # Localisation
                "bbox":         ann.get('bbox'),         # [x, y, w, h] COCO
                "segmentation": ann.get('segmentation'), # {"size":[H,W],"counts":str}
                "area":         ann.get('area', 0.0),
                # DLC-Bench exclusive flags
                "iscrowd":      int(ann.get('iscrowd',     0)),
                "isfake":       int(ann.get('isfake',      0)),
                "isreflected":  int(ann.get('isreflected', 0)),
                "frequency":    ann.get('frequency', ''), # F / C / R
                "size":         ann.get('size',      ''), # S / M / L (object size)
            })

        print(f"   [SUCCESS] Successfully generated {len(self.dlc_samples)} samples!")

    def __len__(self):
        if self.dataset_name == 'dlc_bench':
            return len(self.dlc_samples)
        return len(self.hf_dataset)

    def _get_empty_csv_row(self):
        """Flat 1D schema representing a single row in the exported CSV."""
        return {
            "dataset_name": self.dataset_name.upper(),
            "ann_id": "",
            "img_id": "",
            "image_path": "",    # Original image path for downstream loading
            "mask_size": "",     # Format: "H, W"
            "category": "",
            "caption": "",
            "mask_rle_counts": "", # RLE string
            "bboxes": "",        # Format: "[x, y, w, h]"
            "attributes": "",    # Comma-separated values
            "parts": "",         # Comma-separated values
            # Noise tracking parameters
            "image_noise_type": "",
            "image_severity": 0,
            "bbox_noise_type": "",
            "bbox_severity": 0,
            "mask_noise_type": "",
            "mask_severity": 0,
            # Placeholders for generated model captions
            "pred_keyword": "",
            "pred_phrase": "",
            "pred_detailed": "",
            # DLC-Bench exclusive fields (empty for standard datasets)
            "dlc_frequency":   "",  # F=frequent | C=common | R=rare
            "dlc_obj_size":    "",  # S=small | M=medium | L=large
            "dlc_iscrowd":     0,
            "dlc_isfake":      0,
            "dlc_isreflected": 0,
        }

    # ==========================================
    # Specialized parsing helpers (returning flattened rows)
    # ==========================================
    def _parse_standard(self, raw_ann):
        row = self._get_empty_csv_row()
        row["img_id"] = str(raw_ann.get('img_id', ''))
        row["ann_id"] = str(raw_ann.get('ann_id', ''))
        row["image_path"] = str(raw_ann.get('image', ''))
        row["category"] = str(raw_ann.get('category', ''))
        row["caption"] = str(raw_ann.get('caption', ''))
        
        # Decode RLE mask
        mask_rle = raw_ann.get('mask_rle')
        if mask_rle:
            row["mask_size"] = f"{mask_rle['size'][0]},{mask_rle['size'][1]}"
            counts = mask_rle.get('counts', '')
            if isinstance(counts, bytes):
                counts = counts.decode('utf-8')
            row["mask_rle_counts"] = counts
            
        return row

    def _parse_paco(self, raw_ann):
        row = self._get_empty_csv_row()
        row["img_id"] = str(raw_ann.get('img_id', ''))
        row["ann_id"] = str(raw_ann.get('ann_id', ''))
        row["image_path"] = str(raw_ann.get('image', ''))
        
        row["category"] = str(raw_ann.get('category', raw_ann.get('object_name', '')))
        row["caption"] = str(raw_ann.get('caption', ''))
        
        # PACO contains attribute lists and object parts
        attrs = raw_ann.get('attributes', [])
        row["attributes"] = ", ".join(attrs) if isinstance(attrs, list) else str(attrs)
        
        part = raw_ann.get('part_name')
        row["parts"] = str(part) if part else ""

        # Prioritize object-level masks over generic masks
        mask_rle = raw_ann.get('object_mask_rle', raw_ann.get('mask_rle'))
        if mask_rle:
            row["mask_size"] = f"{mask_rle['size'][0]},{mask_rle['size'][1]}"
            counts = mask_rle.get('counts', '')
            if isinstance(counts, bytes):
                counts = counts.decode('utf-8')
            row["mask_rle_counts"] = counts
            
        return row

    def _parse_dlc_bench(self, sample):
        """
        Accepts an entry from self.dlc_samples (pre-built from COCO JSON).
        Bypasses HF pickle — all fields are pre-resolved (img path, category name, etc.).

        Full schema of DLC-Bench (from diagnostic):
            ann_id, img_id, image_path  — identity
            category, caption           — semantics
            bbox    [x, y, w, h]        — COCO float coords
            segmentation {"size":[H,W], "counts":str}  — RLE mask
            area                        — pixel area
            iscrowd / isfake / isreflected — 0/1 flags
            frequency  F/C/R            — object frequency tier
            size       S/M/L            — object size tier
        """
        row = self._get_empty_csv_row()

        # Identity
        row["img_id"]     = str(sample["img_id"])
        row["ann_id"]     = str(sample["ann_id"])
        row["image_path"] = str(sample["image_path"])

        # Semantics
        row["category"] = sample["category"]
        row["caption"]  = sample["caption"]

        # BBox [x, y, w, h] as fallback; overridden by mask-derived bbox
        bbox = sample.get("bbox")
        if bbox and len(bbox) == 4:
            row["bboxes"] = str([round(float(v), 2) for v in bbox])

        # Segmentation RLE following COCO standard
        seg = sample.get("segmentation")
        if seg and isinstance(seg, dict) and 'size' in seg and 'counts' in seg:
            row["mask_size"] = f"{seg['size'][0]},{seg['size'][1]}"
            counts = seg['counts']
            if isinstance(counts, bytes):
                counts = counts.decode('utf-8')
            row["mask_rle_counts"] = counts

        # DLC-Bench exclusive flags
        row["dlc_frequency"]   = sample.get("frequency",   "")
        row["dlc_obj_size"]    = sample.get("size",        "")
        row["dlc_iscrowd"]     = sample.get("iscrowd",     0)
        row["dlc_isfake"]      = sample.get("isfake",      0)
        row["dlc_isreflected"] = sample.get("isreflected", 0)

        return row

    def _parse_flickr30k(self, raw_ann):
        return self._get_empty_csv_row()

    def xywh_to_xyxy(self, box):
        x, y, w, h = box
        return [x, y, x + w, y + h]

    # ==========================================
    # __getitem__: Load and apply perturbations
    # ==========================================
    def __getitem__(self, idx):

        # ── DLC-Bench: Load image from disk and parse annotations ──────
        if self.dataset_name == 'dlc_bench':
            sample = self.dlc_samples[idx]

            # Load image from disk
            try:
                image = Image.open(sample["image_path"]).convert("RGB")
            except Exception as e:
                print(f"   [ERROR] Failed to load image {sample['image_path']}: {e}")
                image = Image.new('RGB', (224, 224), (0, 0, 0))

            # Parse ra 1 row (DLC-Bench: 1 annotation = 1 sample)
            row = self._parse_dlc_bench(sample)
            raw_anns_parsed = [row]

        # ── Hugging Face datasets: Load from dataset object ───────────
        else:
            sample = self.hf_dataset[idx]

            # Flexible image key lookup ('image' or 'jpg')
            image = sample.get('jpg', sample.get('image'))
            if not isinstance(image, Image.Image):
                image = Image.fromarray(image)
            if image.mode != "RGB":
                image = image.convert("RGB")

            # 2. Read annotations list
            raw_anns = pickle.loads(sample['pickle'])
            if not isinstance(raw_anns, list):
                raw_anns = [raw_anns]

            parser_func = self.parsers[self.dataset_name]
            raw_anns_parsed = [parser_func(raw_ann) for raw_ann in raw_anns]

        rows_list = []
        binary_masks = []
        bboxes = []

        # 3. Parse sample metadata
        for row in raw_anns_parsed:
            
            # Decode RLE mask to binary mask for noise and bbox extraction
            b_mask = None
            if row["mask_rle_counts"] and row["mask_size"]:
                h, w = map(int, row["mask_size"].split(','))
                # Safe handling for string, bytes, and uncompressed list RLEs
                raw_counts = row["mask_rle_counts"]
                if isinstance(raw_counts, str):
                    valid_counts = raw_counts.encode('utf-8')
                else:
                    valid_counts = raw_counts # Keep list intact for pycocotools
                    
                rle_dict = {
                    'size': [h, w],
                    'counts': valid_counts 
                }
                b_mask = mask_util.decode(rle_dict)
            binary_masks.append(b_mask)

            # Derive bounding box from binary mask
            # Prioritize mask-derived bbox for cross-dataset consistency
            # Fallback to annotation bbox if mask is unavailable
            current_bbox = None
            if b_mask is not None:
                current_bbox = mask_util.toBbox(rle_dict).tolist()
            elif row["bboxes"]:
                # Fallback: use parsed bbox
                try:
                    current_bbox = ast.literal_eval(row["bboxes"])
                except (ValueError, SyntaxError):
                    current_bbox = None
            bboxes.append(current_bbox)
            
            rows_list.append(row)

        # 4. Apply configured perturbations
        if self.perturber:
            noisy_img, noisy_bboxes, noisy_masks = self.perturber.apply_to_data(
                image=image, 
                bboxes=bboxes, 
                masks=binary_masks
            )
            image = noisy_img
        else:
            noisy_bboxes = []
            for bbox in bboxes:
                if bbox is not None:
                    noisy_bboxes.append(self.xywh_to_xyxy(bbox))
                else:
                    noisy_bboxes.append(None)
            noisy_masks = binary_masks

        # 5. Update perturbed BBox, RLE mask, and perturbation metadata
        for i, row in enumerate(rows_list):
            # Store bbox coordinates as string "[x, y, w, h]"
            if noisy_bboxes is not None and noisy_bboxes[i] is not None:
                row["bboxes"] = str([round(x, 2) for x in noisy_bboxes[i]])
            
            n_mask = noisy_masks[i] if noisy_masks else None
            
            # Re-encode perturbed mask to RLE string
            if n_mask is not None:
                noisy_mask_f = np.asfortranarray(n_mask).astype(np.uint8)
                encoded_noisy_mask = mask_util.encode(noisy_mask_f)
                
                counts = encoded_noisy_mask['counts']
                if isinstance(counts, bytes):
                    counts = counts.decode('utf-8')
                row["mask_rle_counts"] = counts
                
                # Update mask dimensions
                row["mask_size"] = f"{encoded_noisy_mask['size'][0]},{encoded_noisy_mask['size'][1]}"
            
            # Record perturbation configuration
            if self.perturber:
                row["image_noise_type"] = str(self.perturber.image_noise_type)
                row["image_severity"] = self.perturber.image_severity
                row["bbox_noise_type"] = self.perturber.bbox_noise_type
                row["bbox_severity"] = self.perturber.bbox_severity
                row["mask_noise_type"] = str(self.perturber.mask_noise_type)
                row["mask_severity"] = self.perturber.mask_severity

        return {
            "image": image,           # Transformed image for model inference
            "masks": noisy_masks,     # Target binary masks 
            "bboxes": noisy_bboxes,   # Target bounding boxes
            "csv_rows": rows_list     # Metadata rows for CSV export
        }

In [ ]:
# LVIS
# hf_dataset = load_dataset("nvidia/describe-anything-dataset", "LVIS")['train']
# print(len(hf_dataset))

# COCOStuff
# hf_dataset = load_dataset("nvidia/describe-anything-dataset", "COCOStuff")['train']
# print(len(hf_dataset))

# PACO
hf_dataset = load_dataset("nvidia/describe-anything-dataset", "PACO")['train']
print(len(hf_dataset))


# Note: DLC-Bench is loaded locally from disk; no Hugging Face download required.
# hf_dataset = 1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import copy
import math

def visualize_benchmark_matrix(clean_dataset, start_index, end_index):
    """
    Render 110-test benchmark evaluation matrix for samples in range [start_index, end_index).
    Requirement: clean_dataset must be initialized with perturber=None.
    """
    print(f"\n[VISUALIZATION] Rendering 110-test benchmark matrix for images {start_index} to {end_index-1}...")

    # ==========================================
    # 1. DEFINE 110 BENCHMARK MATRIX TEST SUITES
    # ==========================================
    configs = []

    # a. Clean Baselines (Mask & BBox)
    configs.append({"title": "1. Base (GT Mask)", "params": {}, "mode": "mask"})
    configs.append({"title": "2. Base (GT BBox)", "params": {}, "mode": "bbox"})

    # b. Image Corruptions (15 types x 5 severities)
    img_noises = ['gaussian_noise', 'shot_noise', 'impulse_noise',
                  'defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur',
                  'snow', 'frost', 'fog', 'brightness',
                  'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression']
    for n_type in img_noises:
        for s in range(1, 6):
            configs.append({
                "title": f"Img: {n_type} (Lv{s})",
                "params": {"image_noise_type": n_type, "image_severity": s},
                "mode": "mask"
            })

    # c. Spatial BBox Noise (3 types x 5 severities)
    bbox_noises = ["shift", "upscale", "downscale"]
    for n_type in bbox_noises:
        for s in range(1, 6):
            configs.append({
                "title": f"BBox: {n_type} (Lv{s})",
                "params": {"bbox_noise_type": n_type, "bbox_severity": s},
                "mode": "bbox"
            })

    # d. Mask Perturbations (4 types x 5 severities)
    mask_noises = ["dilation", "erosion", "translation", "boundary_jitter"]
    for n_type in mask_noises:
        for s in range(1, 6):
            configs.append({
                "title": f"Mask: {n_type} (Lv{s})",
                "params": {"mask_noise_type": n_type, "mask_severity": s},
                "mode": "mask"
            })

    # ==========================================
    # 2. RENDER MATRIX FOR EACH SAMPLE
    # ==========================================
    actual_end = min(end_index, len(clean_dataset))
    for idx in range(start_index, actual_end):
        print(f"-> Rendering sample index: {idx}...")

        # Extract clean sample from dataset
        batch = clean_dataset[idx]
        base_img = np.array(batch['image'])
        base_bboxes = batch['bboxes']
        base_masks = batch['masks']

        # Dynamic subplot grid: 5 columns
        n_cols = 5
        n_rows = math.ceil(len(configs) / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))
        axes = axes.flatten()
        fig.suptitle(f"Robustness Benchmark Matrix (110 Tests) - Sample: {idx}", fontsize=28, fontweight='bold', y=0.98)

        for i, cfg in enumerate(configs):
            ax = axes[i]
            params = cfg["params"]
            mode = cfg["mode"]

            # Instantiate AddNoise for current test
            perturber = AddNoise(**params) if params else None

            # Deepcopy sample data to isolate test conditions
            img_c = base_img.copy()
            bboxes_c = copy.deepcopy(base_bboxes) if base_bboxes else None
            masks_c = [m.copy() if m is not None else None for m in base_masks] if base_masks else None
            # print(bboxes_c)
            # Apply perturbation
            if perturber:
                new_bboxes_c = []
                for bbox in bboxes_c:
                    new_bboxes_c.append([bbox[0], bbox[1], bbox[2] - bbox[0], bbox[3] - bbox[1]])
                noisy_img, noisy_bboxes, noisy_masks = perturber.apply_to_data(
                    image=img_c, bboxes=new_bboxes_c, masks=masks_c
                )
            else:
                noisy_img, noisy_bboxes, noisy_masks = img_c, bboxes_c, masks_c

            # Render image
            if not isinstance(noisy_img, np.ndarray):
                noisy_img = np.array(noisy_img)
            ax.imshow(noisy_img)

            # Render region overlay (Mask or BBox)
            if mode == "mask" and noisy_masks:
                for mask in noisy_masks:
                    if mask is None:
                        continue
                    m_np = np.array(mask)
                    # Green translucent mask overlay
                    color_mask = np.zeros((*m_np.shape, 4))
                    color_mask[m_np > 0] = [0, 1, 0, 0.45]
                    ax.imshow(color_mask)

            elif mode == "bbox" and noisy_bboxes:
                for bbox in noisy_bboxes:
                    if bbox is None:
                        continue
                        
                    # 1. Input format: [xmin, ymin, xmax, ymax]
                    x1, y1, x2, y2 = bbox
                    
                    # 2. Compute width (w) and height (h) for matplotlib
                    w = x2 - x1
                    h = y2 - y1
                    
                    # 3. Draw bounding box rectangle
                    rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)

            # Set title and border styling
            ax.set_title(cfg["title"], fontsize=12, pad=6)
            ax.axis('off')

        # Turn off unused subplot axes
        for j in range(len(configs), len(axes)):
            axes[j].axis('off')

        plt.subplots_adjust(wspace=0.05, hspace=0.2, top=0.97)
        plt.show()



# visualize_benchmark_matrix(UniversalDataset(hf_dataset=hf_dataset, dataset_name="paco", perturber=None), 1, 2)

In [ ]:
%%writefile gpt4roi_model.py
import os
import math
import re
from abc import ABCMeta, abstractmethod
from functools import partial
from typing import List, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from mmcv import ops
from mmcv.cnn import ConvModule
from mmengine.model import BaseModule, normal_init
from torch.nn import Linear
from transformers import (
    AutoConfig, AutoModelForCausalLM,
    LlamaConfig, LlamaModel, LlamaForCausalLM,
    CLIPVisionModel, CLIPImageProcessor,
)
from transformers.modeling_outputs import BaseModelOutputWithPast, CausalLMOutputWithPast

# ─── Constants ────────────────────────────────────────────────────────────────
DEFAULT_IMAGE_TOKEN       = "<image>"
DEFAULT_IMAGE_PATCH_TOKEN = "<im_patch>"
DEFAULT_IM_START_TOKEN    = "<im_start>"
DEFAULT_IM_END_TOKEN      = "<im_end>"

class LlavaConfig(LlamaConfig):
    model_type = "llava_custom"

class BaseRoIExtractor(BaseModule, metaclass=ABCMeta):
    def __init__(self, roi_layer, out_channels, featmap_strides, init_cfg=None):
        super().__init__(init_cfg)
        self.roi_layers      = self.build_roi_layers(roi_layer, featmap_strides)
        self.out_channels    = out_channels
        self.featmap_strides = featmap_strides
        self.fp16_enabled    = False

    @property
    def num_inputs(self):
        return len(self.featmap_strides)

    def build_roi_layers(self, layer_cfg, featmap_strides):
        cfg        = layer_cfg.copy()
        layer_type = cfg.pop('type')
        assert hasattr(ops, layer_type)
        layer_cls  = getattr(ops, layer_type)
        return nn.ModuleList(
            [layer_cls(spatial_scale=1 / s, **cfg) for s in featmap_strides]
        )

    def roi_rescale(self, rois, scale_factor):
        cx    = (rois[:, 1] + rois[:, 3]) * 0.5
        cy    = (rois[:, 2] + rois[:, 4]) * 0.5
        w     = rois[:, 3] - rois[:, 1]
        h     = rois[:, 4] - rois[:, 2]
        new_w = w * scale_factor
        new_h = h * scale_factor
        x1    = cx - new_w * 0.5
        x2    = cx + new_w * 0.5
        y1    = cy - new_h * 0.5
        y2    = cy + new_h * 0.5
        return torch.stack((rois[:, 0], x1, y1, x2, y2), dim=-1)

    @abstractmethod
    def forward(self, feats, rois, roi_scale_factor=None):
        pass

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(
            Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim])
        )

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x

def padding_to(inputs, max=300):
    num_padding = max - len(inputs)
    if inputs.dim() > 1:
        padding = inputs.new_zeros(num_padding, *inputs.size()[1:])
    else:
        padding = inputs.new_zeros(num_padding)
    return torch.cat([inputs, padding], dim=0)

def align_tensor(inputs, max_len=None):
    if max_len is None:
        max_len = max(len(item) for item in inputs)
    return torch.stack([padding_to(item, max_len) for item in inputs])

class MLVLFuseModule(nn.Module):
    def __init__(self, input_dims=1024, embed_dims=1024, num_levels=3, num_fuse=4):
        super().__init__()
        self.embed_dims      = embed_dims
        self.num_levels      = num_levels
        self.num_fuse        = num_fuse
        self.input_dims      = input_dims
        self.shuffle_channles = embed_dims // 4
        self.remain_chs      = embed_dims - self.shuffle_channles * 2

        self.fuse_lvl_list = []
        for lvl in range(num_levels):
            top_lvl = min(lvl + 1, num_levels - 1)
            dow_lvl = max(lvl - 1, 0)
            self.fuse_lvl_list.append((lvl, top_lvl, dow_lvl))

        self._init_layers()

    def generate_coordinate(self, featmap_sizes, device='cuda', dtype=torch.float32):
        x_range    = torch.linspace(-1, 1, featmap_sizes[-1], device=device, dtype=dtype)
        y_range    = torch.linspace(-1, 1, featmap_sizes[-2], device=device, dtype=dtype)
        y, x       = torch.meshgrid(y_range, x_range, indexing='ij')
        y          = y.expand([featmap_sizes[0], 1, -1, -1])
        x          = x.expand([featmap_sizes[0], 1, -1, -1])
        return torch.cat([x, y], 1)

    def _init_layers(self):
        self.input_conv = nn.ModuleList([
            nn.Conv2d(self.input_dims + 2, self.embed_dims, 1)
            for _ in range(self.num_levels)
        ])
        self.fuse_convs = nn.ModuleList([
            ConvModule(self.embed_dims, self.embed_dims, 3,
                       stride=1, padding=1, conv_cfg=None,
                       norm_cfg=dict(type='GN', num_groups=64, requires_grad=True))
            for _ in range(self.num_fuse)
        ])

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                normal_init(m, std=0.01)

    def _single_shuffle(self, inputs, conv_module):
        if not isinstance(conv_module, (nn.ModuleList, list)):
            conv_module = [conv_module]
        for single_conv_m in conv_module:
            fused_inputs = []
            for tar_lvl, top_lvl, dow_lvl in self.fuse_lvl_list:
                tar_input  = inputs[tar_lvl]
                top_input  = inputs[top_lvl]
                down_input = inputs[dow_lvl]
                remain     = tar_input[:, :self.remain_chs]
                
                from_top   = top_input[:, self.remain_chs:][:, self.shuffle_channles:]
                from_top   = F.interpolate(from_top.to(torch.float32),
                                           size=tar_input.shape[-2:],
                                           mode='bilinear', align_corners=True).to(tar_input.dtype)
                
                from_down  = down_input[:, self.remain_chs:][:, :self.shuffle_channles]
                from_down  = F.interpolate(from_down.to(torch.float32),
                                           size=tar_input.shape[-2:],
                                           mode='bilinear', align_corners=True).to(tar_input.dtype)
                
                fused_inputs.append(
                    torch.cat([remain, from_top, from_down], dim=1))
            fused_inputs = [single_conv_m(item) for item in fused_inputs]
            inputs = fused_inputs
        return inputs

    def forward(self, inputs):
        new_inputs = []
        for feat in inputs:
            coord_feat = self.generate_coordinate(feat.shape, device=feat.device, dtype=feat.dtype)
            new_inputs.append(torch.cat([feat, coord_feat], dim=1))
        inputs = new_inputs
        inputs = [self.input_conv[lvl](item) for lvl, item in enumerate(inputs)]
        for conv_m in self.fuse_convs:
            inputs = self._single_shuffle(inputs, [conv_m])
        return inputs


class MlvlRoIExtractor(BaseRoIExtractor):
    def __init__(self, roi_layer, out_channels, featmap_strides,
                 embed_dims=1024, stride=1, norm_init=True,
                 fuse_level=3, finest_scale=56, init_cfg=None):
        super().__init__(roi_layer, out_channels, featmap_strides, init_cfg)
        self.embed_dims   = embed_dims
        self.finest_scale = finest_scale
        self.fuse_level   = fuse_level
        self.norm_init    = norm_init

        self.pconvs = nn.ModuleList([
            nn.Conv2d(self.embed_dims, self.embed_dims, 3, stride=1, padding=1)
            for _ in range(self.fuse_level)
        ])
        self.pos_embedd = nn.Sequential(
            nn.Linear(4, 256), nn.ReLU(inplace=True), nn.LayerNorm(256),
            nn.Linear(256, 1024), nn.ReLU(inplace=True), nn.LayerNorm(1024),
        )
        self.updims        = nn.Linear(1024, 4096)
        self.flatten_linear = nn.Linear(
            self.embed_dims * self.roi_layers[0].output_size[0] ** 2, 1024)
        self.norm_init_weights()

    def norm_init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                normal_init(m, 0, 0.01)

    def forward(self, feats, rois, roi_scale_factor=None):
        num_imgs   = len(rois)
        batch_rois = torch.cat(rois, dim=0)
        
        pos_embedd = self.pos_embedd(batch_rois.to(feats[0].dtype))
        out_size   = self.roi_layers[0].output_size
        num_levels = len(feats)

        new_rois = []
        for img_id, single_img_roi in enumerate(rois):
            single_img_roi = single_img_roi * 224
            roi_img_id     = single_img_roi.new_ones(len(single_img_roi)) * img_id
            new_rois.append(torch.cat([roi_img_id[:, None], single_img_roi], dim=1))
        rois_fmt = torch.cat(new_rois)

        roi_feats = feats[0].new_zeros(
            self.fuse_level, rois_fmt.size(0), self.out_channels, *out_size)

        for i in range(num_levels):
            if len(rois_fmt) > 0:
                ori_dtype    = feats[i].dtype
                roi_feats[i] = self.roi_layers[i](
                    feats[i].to(torch.float32),
                    rois_fmt.to(torch.float32)
                ).to(ori_dtype)
            else:
                roi_feats += sum(
                    x.view(-1)[0] for x in self.parameters()) * 0. + feats[i].sum() * 0.

        fuse_roi_feats = sum(self.pconvs[i](roi_feats[i]) for i in range(self.fuse_level))
        fuse_roi_feats = F.relu(fuse_roi_feats)
        fuse_roi_feats = self.flatten_linear(fuse_roi_feats.flatten(1))
        fuse_roi_feats = self.updims(fuse_roi_feats + pos_embedd)

        return [fuse_roi_feats[rois_fmt[:, 0] == i] for i in range(num_imgs)]

class MLVLROIQueryModule(nn.Module):
    def __init__(self, embed_dims=1024, out_dims=4096, num_levels=3):
        super().__init__()
        self.mlvl_fuse = MLVLFuseModule(input_dims=embed_dims,
                                        embed_dims=embed_dims,
                                        num_levels=num_levels,
                                        num_fuse=5)
        strids = [14/8, 14/4, 14/2, 14]
        assert len(strids) == num_levels
        self.roi_align = MlvlRoIExtractor(
            roi_layer=dict(type='RoIAlign', output_size=14, sampling_ratio=2),
            out_channels=embed_dims,
            embed_dims=embed_dims,
            fuse_level=num_levels,
            featmap_strides=strids,
        )

    def forward(self, mlvl_feats, bboxes):
        if mlvl_feats[0].dim() == 3:
            b, _, c = mlvl_feats[0].shape
            h = w   = int(math.sqrt(mlvl_feats[0].shape[1]))
            mlvl_feats = [f.reshape(b, h, w, c).permute(0, 3, 1, 2)
                          for f in mlvl_feats]

        base_shape = mlvl_feats[0].shape[-2:]
        num_level  = len(mlvl_feats)
        to_shape   = [(base_shape[0] * 2**lvl, base_shape[1] * 2**lvl)
                      for lvl in range(num_level)][::-1]

        mlvl_feats = [
            F.interpolate(mlvl_feats[lvl].to(torch.float32), size=to_shape[lvl],
                          mode='bilinear', align_corners=True).to(mlvl_feats[lvl].dtype)
            for lvl in range(num_level)
        ]
        mlvl_feats = self.mlvl_fuse(mlvl_feats)
        return self.roi_align(mlvl_feats, bboxes)

class LlavaLlamaModel(LlamaModel):
    config_class = LlavaConfig

    def __init__(self, config: LlamaConfig):
        super().__init__(config)
        if hasattr(config, "mm_vision_tower"):
            self.vision_tower_name = config.mm_vision_tower
            self.vision_tower      = []
        if hasattr(config, "use_mm_proj"):
            self.mm_projector = nn.Linear(config.mm_hidden_size, config.hidden_size)

    def initialize_vision_tower(self):
        if self.vision_tower_name and len(self.vision_tower) == 0:
            self.vision_tower = [CLIPVisionModel.from_pretrained(self.vision_tower_name)]

def add_spatial_token(tokenizer):
    num_spi_tokens = tokenizer.add_tokens(['<bbox>', '<point>'], special_tokens=True)
    return tokenizer, num_spi_tokens

class SPILlavaLlamaModel(LlavaLlamaModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_level_spi_features = 4
        self.spi_module = MLVLROIQueryModule(embed_dims=1024, out_dims=4096, num_levels=4)

    def forward(self, input_ids=None, attention_mask=None, img_metas=None, bboxes=None, past_key_values=None, inputs_embeds=None, use_cache=None, output_attentions=None, output_hidden_states=None, images=None, position_ids=None, cache_position=None, return_dict=None, **kwargs):
        orig_embeds_params = getattr(self, 'orig_embeds_params', None)

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        vision_tower = getattr(self, 'vision_tower', None)
        if vision_tower is not None and input_ids is not None and (input_ids.shape[1] != 1 or self.training) and images is not None:
            vision_tower = vision_tower[0]
            with torch.no_grad():
                image_forward_outs = vision_tower(images, output_hidden_states=True)
                select_hidden_state_layer = getattr(self.config, 'mm_vision_select_layer', -1)
                select_hidden_state = image_forward_outs.hidden_states[select_hidden_state_layer]
                image_features = select_hidden_state[:, 1:]

                mlvl_spi_features = image_forward_outs.hidden_states[select_hidden_state_layer::-3]
                mlvl_spi_features = mlvl_spi_features[::-1]
                mlvl_spi_features = mlvl_spi_features[-self.num_level_spi_features:]
                mlvl_spi_features = [item[:, 1:] for item in mlvl_spi_features]

            if bboxes is not None and len(bboxes) > 0:
                spi_dev = next(self.spi_module.parameters()).device
                spi_dtype = next(self.spi_module.parameters()).dtype
                
                # Align device & precision
                mlvl_spi_features = [item.to(device=spi_dev, dtype=spi_dtype) if item is not None else None for item in mlvl_spi_features]
                
                # Compute standard RoI features without clamping
                mlvl_spi_features = self.spi_module(mlvl_spi_features, bboxes)
                
                # Project to LLM embedding dimension for token mapping
                mlvl_spi_features = [item.to(inputs_embeds.dtype) if item is not None else None for item in mlvl_spi_features]
            else:
                mlvl_spi_features = [None] * len(input_ids)

            image_features = self.mm_projector(image_features)
            dummy_image_features = torch.zeros(256, 1024, device=inputs_embeds.device, dtype=inputs_embeds.dtype)
            dummy_image_features = self.mm_projector(dummy_image_features)

            new_input_embeds = []
            cur_image_idx    = 0

            for cur_input_ids, cur_input_embeds, spi_feat in zip(input_ids, inputs_embeds, mlvl_spi_features):
                if (cur_input_ids == vision_tower.config.im_patch_token).sum() == 0:
                    cur_input_embeds = cur_input_embeds + (0. * dummy_image_features).sum().to(cur_input_embeds.device)
                    new_input_embeds.append(cur_input_embeds)
                    cur_image_idx += 1
                    continue

                if vision_tower.config.use_im_start_end:
                    if (cur_input_ids == vision_tower.config.im_start_token).sum() != (cur_input_ids == vision_tower.config.im_end_token).sum():
                        raise ValueError('Number of im_start_token and im_end_token must match.')

                    image_start_tokens = torch.where(cur_input_ids == vision_tower.config.im_start_token)[0]

                    for image_start_token_pos in image_start_tokens:
                        image_start_token_pos = image_start_token_pos.item()
                        cur_image_features = image_features[cur_image_idx].to(device=cur_input_embeds.device)
                        num_patches = cur_image_features.shape[0]

                        if orig_embeds_params is not None:
                            cur_new_input_embeds = torch.cat((
                                cur_input_embeds[:image_start_token_pos].detach(),
                                cur_input_embeds[image_start_token_pos:image_start_token_pos + 1],
                                cur_image_features,
                                cur_input_embeds[image_start_token_pos + num_patches + 1:image_start_token_pos + num_patches + 2],
                                cur_input_embeds[image_start_token_pos + num_patches + 2:].detach(),
                            ), dim=0)
                        else:
                            cur_new_input_embeds = torch.cat((
                                cur_input_embeds[:image_start_token_pos + 1],
                                cur_image_features,
                                cur_input_embeds[image_start_token_pos + num_patches + 1:],
                            ), dim=0)

                        if spi_feat is not None:
                            tokenizer_ref = getattr(self, 'tokenizer', None)
                            if tokenizer_ref is None:
                                raise RuntimeError("Tokenizer is not attached to model. Run setup cells first.")
                            
                            bbox_tok_id = tokenizer_ref.convert_tokens_to_ids(['<bbox>'])[0]
                            spi_mask = (cur_input_ids == bbox_tok_id)
                            n_assign = min(spi_mask.sum().item(), spi_feat.shape[0])
                            
                            if n_assign > 0:
                                spi_embeds = torch.zeros_like(cur_new_input_embeds)
                                mask_indices = torch.where(spi_mask)[0][:n_assign]
                                exact_mask = torch.zeros_like(spi_mask, dtype=torch.bool)
                                exact_mask[mask_indices] = True
                                
                                spi_embeds[exact_mask] = spi_feat[:n_assign].to(device=spi_embeds.device, dtype=spi_embeds.dtype)
                                cur_new_input_embeds = cur_new_input_embeds * (~exact_mask).to(cur_input_embeds.dtype)[:, None] + spi_embeds
                        else:
                            tokenizer_ref = getattr(self, 'tokenizer', None)
                            if tokenizer_ref is not None:
                                bbox_tok_id = tokenizer_ref.convert_tokens_to_ids(['<bbox>'])[0]
                                assert (cur_input_ids == bbox_tok_id).sum() == 0, "Encountered <bbox> token but spi_feat is None!"

                        cur_image_idx += 1
                    new_input_embeds.append(cur_new_input_embeds)

                else:
                    cur_image_features = image_features[cur_image_idx].to(device=cur_input_embeds.device)
                    num_patches  = cur_image_features.shape[0]
                    masked_indices   = torch.where(cur_input_ids == vision_tower.config.im_patch_token)[0]
                    mask_index_start = masked_indices[0].item()

                    if orig_embeds_params is not None:
                        cur_new_input_embeds = torch.cat((
                            cur_input_embeds[:mask_index_start].detach(),
                            cur_image_features,
                            cur_input_embeds[mask_index_start + num_patches:].detach(),
                        ), dim=0)
                    else:
                        cur_new_input_embeds = torch.cat((
                            cur_input_embeds[:mask_index_start],
                            cur_image_features,
                            cur_input_embeds[mask_index_start + num_patches:],
                        ), dim=0)
                    new_input_embeds.append(cur_new_input_embeds)
                    cur_image_idx += 1

            inputs_embeds = torch.stack([e.to(new_input_embeds[0].device) for e in new_input_embeds], dim=0)

        return super(LlavaLlamaModel, self).forward(
            input_ids=None, attention_mask=attention_mask,
            past_key_values=past_key_values, inputs_embeds=inputs_embeds,
            use_cache=use_cache, output_attentions=output_attentions,
            output_hidden_states=output_hidden_states, return_dict=return_dict,
            position_ids=position_ids, cache_position=cache_position,
        )

class LlavaLlamaForCausalLM(LlamaForCausalLM):
    config_class = LlavaConfig

    def __init__(self, config):
        super(LlamaForCausalLM, self).__init__(config)
        self.model   = LlavaLlamaModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.post_init()

class SPILlavaMPTForCausalLM(LlavaLlamaForCausalLM):
    def __init__(self, config):
        super(LlavaLlamaForCausalLM, self).__init__(config)
        self.model   = SPILlavaLlamaModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.post_init()

    def forward(self, *args, img_metas=None, bboxes=None, images=None, **kwargs):
        self.model.orig_forward = self.model.forward
        self.model.forward = partial(self.model.orig_forward,
                                     img_metas=img_metas,
                                     bboxes=bboxes,
                                     images=images)
        out = super().forward(*args, **kwargs)
        self.model.forward = self.model.orig_forward
        return out

    def initialize_vision_tokenizer(self, mm_use_im_start_end, tokenizer, device, tune_mm_mlp_adapter=False, pretrain_mm_mlp_adapter=None):
        vision_config = self.get_model().vision_tower[0].config
        vision_config.use_im_start_end = mm_use_im_start_end
        tokenizer.add_tokens([DEFAULT_IMAGE_PATCH_TOKEN], special_tokens=True)
        self.resize_token_embeddings(len(tokenizer))

        tokenizer, num_spi_tokens = add_spatial_token(tokenizer)
        if mm_use_im_start_end:
            num_new_tokens = tokenizer.add_tokens(
                [DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN], special_tokens=True)
            self.resize_token_embeddings(len(tokenizer))
            vision_config.im_start_token, vision_config.im_end_token = \
                tokenizer.convert_tokens_to_ids([DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN])

            num_new_tokens = num_new_tokens + num_spi_tokens
            if num_new_tokens > 0:
                input_embeddings  = self.get_input_embeddings().weight.data
                output_embeddings = self.get_output_embeddings().weight.data
                input_embeddings[-num_new_tokens:]  = input_embeddings[:-num_new_tokens].mean(0, keepdim=True)
                output_embeddings[-num_new_tokens:] = output_embeddings[:-num_new_tokens].mean(0, keepdim=True)

            if tune_mm_mlp_adapter:
                self.get_model().orig_embeds_params = [self.get_input_embeddings().weight.data.clone().to(device=device)]
                for p in self.get_input_embeddings().parameters(): p.requires_grad = True
                for p in self.get_output_embeddings().parameters(): p.requires_grad = False

            if pretrain_mm_mlp_adapter:
                mm_projector_weights = torch.load(pretrain_mm_mlp_adapter, map_location='cpu')
                embed_tokens_weight  = mm_projector_weights['model.embed_tokens.weight']
                num_new_tokens_orig  = num_new_tokens - num_spi_tokens
                if input_embeddings.shape == embed_tokens_weight.shape:
                    input_embeddings[-num_new_tokens_orig:] = embed_tokens_weight[-num_new_tokens_orig:]
                elif embed_tokens_weight.shape[0] == num_new_tokens_orig:
                    input_embeddings[-num_new_tokens_orig:] = embed_tokens_weight
                else:
                    raise ValueError(f'Unexpected embed_tokens_weight shape.')

        vision_config.im_patch_token = tokenizer.convert_tokens_to_ids([DEFAULT_IMAGE_PATCH_TOKEN])[0]
        vision_config.bbox_token     = tokenizer.convert_tokens_to_ids(['<bbox>'])[0]
        vision_config.point_token    = tokenizer.convert_tokens_to_ids(['<point>'])[0]
        for m in self.modules(): m.tokenizer = tokenizer

AutoConfig.register("llava_custom", LlavaConfig, exist_ok=True)
AutoModelForCausalLM.register(LlavaConfig, SPILlavaMPTForCausalLM, exist_ok=True)

In [ ]:
"""
Usage:
 python scripts/apply_delta.py  --base ./llama-7b-hf  --target ./gpt4roi-7b  --delta ./gpt4roi-7b-delta
"""
import argparse

import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

from gpt4roi_model import SPILlavaMPTForCausalLM



def apply_delta(base_model_path, target_model_path, delta_path):
    print("Loading base model")
    base = AutoModelForCausalLM.from_pretrained(
        base_model_path, torch_dtype=torch.float16, low_cpu_mem_usage=True)

    print("Loading delta")
    delta = SPILlavaMPTForCausalLM.from_pretrained(delta_path, torch_dtype=torch.float16, low_cpu_mem_usage=True)
    # Initialize vision tower after model is loaded (avoids nested from_pretrained with meta device)
    if hasattr(delta.model, 'initialize_vision_tower'):
        delta.model.initialize_vision_tower()
    delta_tokenizer = AutoTokenizer.from_pretrained(delta_path)

    print("Applying delta")
    for name, param in tqdm(delta.state_dict().items(), desc="Applying delta"):
        if name not in base.state_dict():
            if name in ['model.mm_projector.weight', 'model.mm_projector.bias']:
                continue
            elif "spi_module" in name:
                continue
            else:
                raise NameError
        if param.data.shape == base.state_dict()[name].shape:
            param.data += base.state_dict()[name]
        else:
            assert name in ['model.embed_tokens.weight', 'lm_head.weight'], \
                f'{name} dimension mismatch: {param.data.shape} vs {base.state_dict()[name].shape}'
            bparam = base.state_dict()[name]
            param.data[:bparam.shape[0], :bparam.shape[1]] += bparam

    print("Saving target model")
    delta.save_pretrained(target_model_path)
    delta_tokenizer.save_pretrained(target_model_path)




# apply_delta('baffo32/decapoda-research-llama-7B-hf', 'GPT4Roi_merge', 'shilongz/GPT4RoI-7B-delta-V0')

In [ ]:
# ============================================================
# CELL 5: LOAD MODEL + MANUAL VISION TOKENIZER INIT (FIXED)
# ============================================================
import os
import torch
from transformers import AutoTokenizer, CLIPImageProcessor
from accelerate.hooks import remove_hook_from_module

from gpt4roi_model import (
    SPILlavaMPTForCausalLM,
    DEFAULT_IMAGE_PATCH_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
)

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# Note: Verify MODEL_PATH matches your Kaggle input directory
MODEL_PATH = "/kaggle/input/models/koahle/gpt4roi/pytorch/default/1/GPT4Roi_merge"
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE      = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)

print("Loading model weights (device_map='auto') ...")
model = SPILlavaMPTForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    device_map="auto",
)

print("Initializing vision tower ...")
if hasattr(model.model, "initialize_vision_tower"):
    model.model.initialize_vision_tower()

print("Moving vision tower to GPU ...")
vt = model.model.vision_tower
if isinstance(vt, list) and len(vt) > 0:
    model.model.vision_tower = [vt[0].to(DEVICE, dtype=DTYPE)]

model.eval()

# ══ DEVICE ALLOCATION AND PRECISION MANAGEMENT ═════════════════════════════
vt_device = next(model.model.vision_tower[0].parameters()).device

if hasattr(model.model, 'spi_module'):
    # 1. Remove accelerate hooks for manual device control
    remove_hook_from_module(model.model.spi_module, recurse=True)

    # 2. Maintain float32 precision for numerical stability
    model.model.spi_module = model.model.spi_module.to(device=vt_device, dtype=torch.float32)
    print(f"  spi_module → {vt_device} (Locked at float32 for numerical stability)  ✅")
# ════════════════════════════════════════════════════════════════════════

print("Setting up vision tokens ...")
vcfg = model.model.vision_tower[0].config

new_tokens = []
for tok in [DEFAULT_IMAGE_PATCH_TOKEN, DEFAULT_IM_START_TOKEN,
            DEFAULT_IM_END_TOKEN, "<bbox>", "<point>"]:
    if tok not in tokenizer.get_vocab():
        new_tokens.append(tok)

if new_tokens:
    tokenizer.add_tokens(new_tokens, special_tokens=True)
    model.resize_token_embeddings(len(tokenizer))
    with torch.no_grad():
        ie = model.get_input_embeddings().weight
        oe = model.get_output_embeddings().weight
        n  = len(new_tokens)
        ie[-n:] = ie[:-n].to(ie.device).mean(0, keepdim=True)
        oe[-n:] = oe[:-n].to(oe.device).mean(0, keepdim=True)

vcfg.im_patch_token = tokenizer.convert_tokens_to_ids([DEFAULT_IMAGE_PATCH_TOKEN])[0]
vcfg.im_start_token = tokenizer.convert_tokens_to_ids([DEFAULT_IM_START_TOKEN])[0]
vcfg.im_end_token   = tokenizer.convert_tokens_to_ids([DEFAULT_IM_END_TOKEN])[0]
vcfg.bbox_token     = tokenizer.convert_tokens_to_ids(["<bbox>"])[0]
vcfg.point_token    = tokenizer.convert_tokens_to_ids(["<point>"])[0]
vcfg.use_im_start_end = True

for m in model.modules():
    m.tokenizer = tokenizer

# ── Cache vcfg directly in model object for modular access in infer() ──
model._vcfg = vcfg

vt0 = model.model.vision_tower[0]
print(f"\n  vision_tower device : {next(vt0.parameters()).device}")
print(f"  mm_projector device : {next(model.model.mm_projector.parameters()).device}")
print(f"  im_patch_token      : {vcfg.im_patch_token}  → '{tokenizer.decode([vcfg.im_patch_token])}'")
print(f"  im_start_token      : {vcfg.im_start_token}  → '{tokenizer.decode([vcfg.im_start_token])}'")
print(f"  im_end_token        : {vcfg.im_end_token}  → '{tokenizer.decode([vcfg.im_end_token])}'")
print(f"  bbox_token          : {vcfg.bbox_token}  → '{tokenizer.decode([vcfg.bbox_token])}'")
print(f"  vocab size          : {len(tokenizer)}")

try:
    clip_name = vcfg._name_or_path
    if not clip_name or "/" not in clip_name:
        raise ValueError
except Exception:
    clip_name = "openai/clip-vit-large-patch14"

# ── Standard CLIPImageProcessor with default center cropping ──
# CLIPImageProcessor default pipeline: resize -> center_crop 224x224 -> normalize
# Preprocessing matches official GPT4RoI training pipeline
image_processor = CLIPImageProcessor.from_pretrained(clip_name)
print(f"\n  image_processor     : {clip_name}")
print("\n✅ Ready to infer.")

In [ ]:
# ============================================================
# CELL 6: IMAGE LOADER UTILITY
# ============================================================
import requests
import io
from PIL import Image


def load_image(source: str) -> Image.Image:
    """
    Load PIL Image from:
      - URL  : "https://..."
      - Path : "/path/to/file.jpg"
      - PIL  : return directly if already an Image.Image
    """
    if isinstance(source, Image.Image):
        return source.convert("RGB")
    if isinstance(source, str) and source.startswith("http"):
        resp = requests.get(source, timeout=15)
        resp.raise_for_status()
        return Image.open(io.BytesIO(resp.content)).convert("RGB")
    return Image.open(source).convert("RGB")

In [ ]:
# ============================================================
# GPT4RoI Regional Inference Helper
# ============================================================
import gc
import re
import torch
import torch.nn.functional as F
from functools import partial
from transformers import StoppingCriteria, StoppingCriteriaList
from PIL import Image
import requests
from io import BytesIO

import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")


class StopAtStringCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length, stop_string="###"):
        self.tokenizer    = tokenizer
        self.stop_string  = stop_string
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] <= self.prompt_length:
            return False
        new_text = self.tokenizer.decode(input_ids[0][self.prompt_length:])
        return self.stop_string in new_text


@torch.inference_mode()
def infer(
    image,
    question: str,
    bboxes=None,
    *,
    max_new_tokens: int = 512,
    temperature: float  = 0.1,
    top_p: float        = 0.1,
) -> str:
    # ── 1. Image Loading ──────────────────────────────────────────────────
    pil_img = image

    width, height = pil_img.size

    # ── 2. Preprocess Image with CLIPImageProcessor (resize -> center_crop -> normalize)
    # Preserves default crop without manual post-interpolation
    # Aligned with official training configuration
    image_tensor = image_processor(pil_img, return_tensors="pt")["pixel_values"][0]
    # Output: tensor shape [3, 224, 224] normalized by CLIP statistics

    # ── 3. Format Bounding Box in float32 ───────────────────────────────────
    # spi_module uses float32; bboxes share matching dtype
    # ── Format Single Target Bounding Box ───────────────────────────────────
    vt_dev = next(model.model.vision_tower[0].parameters()).device
    bbox_tensor = None
    
    if bboxes is not None:
        # If bboxes is flat list [x1, y1, x2, y2], wrap into nested list [[x1, y1, x2, y2]]
        # Ensures consistent indexing for bounding box iterations
        if isinstance(bboxes, (list, tuple)) and not isinstance(bboxes[0], (list, tuple)):
            bboxes = [bboxes]
        
        # Select primary target box if multiple boxes provided
        target_box = bboxes[0] 
        
        # Normalize box coordinates
        norm_box = [
            target_box[0] / width, 
            target_box[1] / height, 
            target_box[2] / width, 
            target_box[3] / height
        ]
        
        # Create tensor with shape [1, 4]
        t = torch.tensor([norm_box], dtype=torch.float32, device=vt_dev)
        
        # Package into list expected by model forward pass
        bbox_tensor = [t]

    # ── 4. Prompt Formatting: Convert region tags into region1 <bbox> ─────────
    formatted_question = re.sub(r'<region(\d+)>', r'region\g<1> <bbox>', question)
    formatted_question = re.sub(r'\<(\d+)\>',    r'region\g<1> <bbox>', formatted_question)
    formatted_question = formatted_question.replace('<>', '<bbox>')

    # ── 5. Construct Prompt following GPT4RoI / LLaVA Conversation Format ───
    #
    # Official conversation format (from llava/conversation.py):
    #   {system}###Human: <im_start><im_patch x256><im_end>\n{question}###Assistant:
    #
    # Image tokens placed directly within Human conversation turn
    # Matches pre-training conversational structure
    #
    system_msg = (
        "A chat between a curious human and an artificial intelligence assistant. "
        "The assistant gives helpful, detailed, and polite answers to the human's questions."
    )

    embed_dev = next(model.get_input_embeddings().parameters()).device
    vcfg      = model._vcfg  # cached from setup

    im_start_id = vcfg.im_start_token
    im_patch_id = vcfg.im_patch_token
    im_end_id   = vcfg.im_end_token

    # Tokenize system message + "###Human: " (with BOS)
    text_before = f"{system_msg}###Human: "
    ids_before  = tokenizer(text_before, return_tensors="pt",
                            add_special_tokens=True).input_ids[0]

    # Generate 256 image patch tokens following "Human: "
    # 224/14 = 16 → 16×16 = 256 patches (CLIP ViT-L/14)
    image_ids = torch.tensor(
        [im_start_id] + [im_patch_id] * 256 + [im_end_id],
        dtype=torch.long,
    )

    # Tokenize question turn + "###Assistant:" (without BOS)
    # Append newline after image tokens matching training specification
    text_after = f"\n{formatted_question}###Assistant:"
    ids_after  = tokenizer(text_after, return_tensors="pt",
                           add_special_tokens=False).input_ids[0]

    # Concatenate: [BOS + system + Human:] + [im_start + patch*256 + im_end] + [\n + question + ###Assistant:]
    input_ids    = torch.cat([ids_before, image_ids, ids_after], dim=0).unsqueeze(0).to(embed_dev)
    prompt_length = input_ids.shape[1]

    # ── 6. Generation Configuration and Forward Pass ───────────────────────
    stopping_criteria = StoppingCriteriaList([
        StopAtStringCriteria(tokenizer, prompt_length, "###")
    ])

    gc.collect()
    torch.cuda.empty_cache()

    model.orig_forward = model.forward
    model.forward = partial(
        model.orig_forward,
        img_metas=[None],
        bboxes=bbox_tensor,
    )

    try:
        with torch.amp.autocast(device_type="cuda"):
            output_ids = model.generate(
                input_ids,
                images=image_tensor.unsqueeze(0).half().cuda(),
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                stopping_criteria=stopping_criteria,
            )
    finally:
        model.forward = model.orig_forward

    # ── 7. Output Decoding and Post-processing ──────────────────────────────
    n_diff = (input_ids != output_ids[:, :prompt_length]).sum().item()
    if n_diff > 0:
        print(f"[Warning] {n_diff} output_ids differ from input_ids")

    response = tokenizer.batch_decode(
        output_ids[:, prompt_length:], skip_special_tokens=True
    )[0].strip()

    # Clean up stop tokens and trailing whitespace
    if response.endswith("###"):
        response = response[:-3]
    if "###" in response:
        response = response.split("###")[0]
    response = response.replace("Assistant: ", "").replace("Assistant:", "").strip()

    return response

In [ ]:
from torch.utils.data import DataLoader, Subset
import gc
import pandas as pd
if __name__ == "__main__":


    print("1. Initializing perturbation engine...")
    # NOTE: To evaluate on CLEAN baseline data,
    # set all noise type parameters below to None.

    image_noise_type = None
    image_severity   = 3
    bbox_noise_type  = None
    bbox_severity    = 2
    mask_noise_type  = None
    mask_severity    = 1

    perturber = AddNoise(
        # ==========================================================
        # AXIS 1: FULL-IMAGE CORRUPTIONS (ImageNet-C)
        # ==========================================================
        # Supported: 'gaussian_noise', 'shot_noise', 'impulse_noise',
        #         'defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur',
        #         'snow', 'frost', 'fog', 'brightness',
        #         'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression'
        image_noise_type=image_noise_type,
        # Severity level: from 1 (mild) to 5 (severe)
        image_severity=image_severity,

        # ==========================================================
        # AXIS 2: SPATIAL BBOX PERTURBATIONS (Shift, Upscale, Downscale)
        # ==========================================================
        # Supported:
        # 1. "upscale" (Context expansion)
        # 2. "downscale" (Object truncation)
        # 3. "shift" (Center translation / Misalignment)
        bbox_noise_type=bbox_noise_type,
        # Perturbation severity: from 1 to 5
        bbox_severity=bbox_severity,

        # ==========================================================
        # AXIS 3: MASK MORPHOLOGICAL PERTURBATIONS
        # ==========================================================
        # Supported:
        # 1. "dilation" (Boundary dilation)
        # 2. "erosion" (Boundary erosion)
        # 3. "translation" (Mask translation)
        # 4. "boundary_jitter" (Edge contour noise)
        mask_noise_type=mask_noise_type,
        # Mask destruction severity: from 1 to 5
        mask_severity=mask_severity,
    )

    dataset_name = "paco" # options: "lvis", "dlc_bench", "cocostuff"
    print("2. Initializing UniversalDataset...")
    if dataset_name == "dlc_bench":
        local_path = '/kaggle/input/datasets/ckhangnguyn/dlc-bench/DLC-Bench'
    else:
        local_path = None
    my_dataset = UniversalDataset(hf_dataset=hf_dataset, dataset_name=dataset_name, perturber=perturber, local_path = local_path)

    # ---------------------------------------------------------
    # STEP 2: DATASET CHUNKING (Prevents Kaggle runtime timeouts)
    # ---------------------------------------------------------
    start_index      = 0     # Start sample index
    end_index        = 6    # End sample index
    actual_end_index = min(end_index, len(my_dataset))

    subset_indices = list(range(start_index, actual_end_index))
    my_subset      = Subset(my_dataset, subset_indices)
    dataloader     = DataLoader(my_subset, batch_size=1, shuffle=False, collate_fn=lambda x: x[0])

    # ---------------------------------------------------------
    # STEP 3: INPUT MODALITY SELECTION
    # ---------------------------------------------------------
    # 'mask': Ground-truth or perturbed mask -> DAM (Upper Bound)
    # 'bbox': Bounding box -> SAM -> Mask -> DAM
    inference_approach = "bbox"

    print(f"-> Selected extraction modality: {inference_approach.upper()}")
    print(f"-> Processing images {start_index} to {actual_end_index - 1}. Total: {len(my_subset)} images.")

    all_csv_rows = []

    # ---------------------------------------------------------
    # STEP 4: MAIN INFERENCE LOOP
    # ---------------------------------------------------------
    print("3. Starting model inference...")
    with torch.no_grad():
        for idx, batch in enumerate(tqdm(dataloader, desc="Processing images")):
            image        = batch["image"]
            csv_rows     = batch["csv_rows"]
            current_img_id = csv_rows[0]["img_id"] if csv_rows else f"Index_{idx}"

            if inference_approach == "mask":
                regions_to_use = batch["masks"]
            else:  # "bbox"
                regions_to_use = batch["bboxes"]

            pred_detailed = []

            # FIX 1: enumerate so we have i for the warning message
            # FIX 2: loop variable renamed to bbox (was mismatched with `region`)
            for i, bbox in enumerate(regions_to_use):

                # FIX 3: check `bbox is None`, not the undefined `region`
                if bbox is None:
                    print(
                        f"\n[WARNING] Image '{current_img_id}' - "
                        f"Skipping region #{i + 1} due to missing/corrupted "
                        f"{inference_approach.upper()}!"
                    )
                    pred_detailed.append("")
                    continue

                # FIX 4: capture return dict in a temp variable so the list
                #         is not overwritten; extract scalar values from the dict.
                detail_result = infer(image,
                                      "Describe the <region1> in detail.",
                                    bboxes=bbox,)
    
                # FIX 6: append the scalar strings, not the raw dicts
                pred_detailed.append(detail_result)

            for i, row in enumerate(csv_rows):
                row["pred_detailed"] = pred_detailed[i] if i < len(pred_detailed) else ""
                all_csv_rows.append(row)

            # CLEANUP PER IMAGE
            # FIX 7: removed `pred_phrases` from del — it was never defined
            del regions_to_use, pred_detailed, image
            gc.collect()
            torch.cuda.empty_cache()

    # ---------------------------------------------------------
    # STEP 5: EXPORT PREDICTIONS TO CSV FOR BENCHMARK EVALUATORS
    # ---------------------------------------------------------
    print("4. Exporting results to CSV...")
    df = pd.DataFrame(all_csv_rows)

    noise_label    = perturber.bbox_noise_type if inference_approach == "bbox" else perturber.mask_noise_type
    severity_label = perturber.bbox_severity   if inference_approach == "bbox" else perturber.mask_severity
    noise_str      = f"{noise_label}_sev{severity_label}" if noise_label else "clean"

    csv_filename = (
        f"gpt4roi_{dataset_name}_{inference_approach}_{noise_str}"
        f"_{start_index}_to_{actual_end_index}.csv"
    )

    df.to_csv(csv_filename, index=False, encoding="utf-8-sig")
    print(f"-> Done! Saved {len(df)} region annotations to {csv_filename}")
    
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # Optional: print memory summary to verify it's cleared
        print(torch.cuda.memory_summary(device=None, abbreviated=False))
    
    print("Model successfully unloaded and VRAM cleared.")